# Bài toán phân loại: Dự đoán khách hàng rời bỏ dịch vụ

            ## Mục tiêu
            - Xây dựng mô hình phân loại dự đoán biến `Churn`.
            - Tìm ra những yếu tố liên quan mạnh đến khả năng rời bỏ.
            - Đánh giá mô hình bằng nhiều metric thay vì chỉ dùng accuracy.

            ## Nguồn dữ liệu
            - Bộ dữ liệu `Telco-Customer-Churn.csv`.
            - Nguồn tham khảo: IBM Sample Data / IBM Skills community.


In [ ]:
from pathlib import Path
            import sys

            ROOT_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
            if str(ROOT_DIR) not in sys.path:
                sys.path.insert(0, str(ROOT_DIR))

            import pandas as pd
            from IPython.display import Image, display

            from ml_coursework.churn import (
                CATEGORICAL_FEATURES,
                DROP_COLUMNS,
                NUMERIC_FEATURES,
                TARGET_COLUMN,
                run_analysis,
            )
            from ml_coursework.data_loader import load_churn_data

            df = load_churn_data()
            print(f"Kích thước dữ liệu: {df.shape}")
            df.head()


### Giải thích khối code import và nạp dữ liệu

            Khối code trên giúp notebook dùng lại code chính của dự án thay vì viết lại từ đầu:
            - `ROOT_DIR` giúp Python tìm được package `ml_coursework`.
            - Các hằng số `TARGET_COLUMN`, `NUMERIC_FEATURES`, `CATEGORICAL_FEATURES` được import từ file code chính để tránh sai lệch giữa notebook và chương trình chạy thật.
            - `load_churn_data()` nạp dữ liệu từ `data/raw/Telco-Customer-Churn.csv`.
            - `run_analysis()` là hàm chạy toàn bộ pipeline phân loại churn.

            Đây là cách tổ chức tốt khi nộp bài vì notebook đóng vai trò trình bày, còn logic chính vẫn nằm trong source code có cấu trúc.


## 1. Khảo sát dữ liệu ban đầu

            Ở bước này cần kiểm tra:
            - biến mục tiêu có cân bằng hay không;
            - cột nào là định danh và cột nào cần đưa vào mô hình;
            - dữ liệu thiếu, định dạng sai kiểu, đặc biệt là `TotalCharges`.


In [ ]:
display(df.info())
            print("\nSố dòng trùng lặp:", df.duplicated().sum())
            print("\nGiá trị thiếu theo cột:")
            display(df.isna().sum().to_frame("missing_count").T)
            print("\nPhân bố churn:")
            display(df[TARGET_COLUMN].value_counts(normalize=True).rename("ratio").to_frame())


### Cách đọc kết quả khảo sát dữ liệu churn

            - `df.info()` dùng để phát hiện các cột bị sai kiểu dữ liệu. Trong bộ churn, `TotalCharges` thường cần chú ý vì có thể bị đọc thành chuỗi.
            - Phân bố `Churn` cho biết dữ liệu có lệch lớp hay không. Nếu lớp churn ít hơn nhiều, accuracy một mình sẽ không đủ để đánh giá.
            - Kiểm tra missing và trùng lặp giúp đảm bảo dữ liệu đầu vào sạch trước khi đưa vào pipeline.

            Đây là bước quan trọng vì bài churn là bài phân loại có yếu tố nghiệp vụ: bỏ sót khách sắp rời bỏ có thể gây thiệt hại cho doanh nghiệp.


## 2. Định nghĩa biến và tiền xử lý

            Lựa chọn xử lý:
            - Loại `customerID` vì đây chỉ là mã định danh.
            - Chuyển `TotalCharges` sang số; giá trị rỗng sẽ được xem là missing.
            - Dùng `class_weight='balanced'` cho các mô hình hỗ trợ vì tập dữ liệu có lệch lớp.


In [ ]:
print("Cột mục tiêu:", TARGET_COLUMN)
            print("Cột loại bỏ:", DROP_COLUMNS)
            print("Biến số:", NUMERIC_FEATURES)
            print("Biến phân loại:", CATEGORICAL_FEATURES)


### Giải thích lựa chọn biến đầu vào churn

            - `customerID` bị loại bỏ vì chỉ là mã định danh, không phải đặc trưng hành vi.
            - Các biến số như `tenure`, `MonthlyCharges`, `TotalCharges` phản ánh thời gian sử dụng và chi phí của khách hàng.
            - Các biến phân loại như `Contract`, `PaymentMethod`, `InternetService`, `TechSupport` phản ánh quan hệ dịch vụ giữa khách hàng và nhà mạng.
            - `Churn` được mã hóa thành 0/1 để mô hình phân loại học được.

            Cách chia biến này giúp mô hình vừa học được thông tin định lượng, vừa học được khác biệt giữa các nhóm khách hàng.


## 3. EDA và huấn luyện mô hình

            Mô hình được so sánh:
            - `Logistic Regression`
            - `Random Forest`
            - `Gradient Boosting`

            Sau khi chọn được mô hình tốt nhất, notebook còn thực hiện thêm:
            - `hyperparameter tuning`
            - `decision threshold tuning` trên tập validation

            Bước threshold tuning quan trọng với bài toán churn vì ngưỡng mặc định `0.5`
            không phải lúc nào cũng cho `F1-score` tốt nhất.

            Metric được theo dõi:
            - `Accuracy`
            - `Precision`
            - `Recall`
            - `F1-score`
            - `ROC-AUC`

            Notebook cũng thêm `Dummy Classifier` để làm baseline và phân tích threshold
            để cho thấy lý do tại sao ta không nên giữ ngưỡng mặc định `0.5`.

            Hàm `run_analysis()` trong bài churn thực hiện các bước:
            - nạp và làm sạch dữ liệu;
            - chuyển `TotalCharges` sang số;
            - mã hóa biến mục tiêu `Churn`;
            - chia train/test có `stratify=y` để giữ tỷ lệ lớp;
            - xây dựng pipeline tiền xử lý;
            - so sánh mô hình bằng cross-validation;
            - tối ưu siêu tham số;
            - tìm decision threshold tối ưu theo F1-score;
            - đánh giá trên tập test và lưu toàn bộ kết quả.


In [ ]:
churn_results = run_analysis()
            print("Mô hình tốt nhất:", churn_results.best_model_name)
            print("Bộ tham số tốt nhất:")
            display(pd.DataFrame([churn_results.best_params]))
            print("Ngưỡng dự đoán tối ưu:", round(churn_results.best_threshold, 4))

            print("\nBảng so sánh cross-validation:")
            display(churn_results.cv_results.round(4))

            print("\nKết quả trên tập test:")
            display(churn_results.tuned_metrics.round(4))

            print("\nSo sánh trước và sau threshold tuning:")
            display(churn_results.optimization_comparison.round(4))

            print("\nClassification report:")
            display(churn_results.classification_report_df.round(4))

            print("\nTop đặc trưng quan trọng:")
            display(churn_results.feature_importance.round(4))

            print("\nPhân tích lỗi theo loại hợp đồng:")
            display(churn_results.error_analysis.round(4))


### Cách đọc bảng kết quả Churn

            - `cv_results`: so sánh các mô hình bằng accuracy, precision, recall, F1 và ROC-AUC.
            - `best_params`: cho biết bộ siêu tham số tốt nhất của mô hình được chọn.
            - `best_threshold`: là ngưỡng xác suất tối ưu. Khách hàng có xác suất churn lớn hơn ngưỡng này sẽ được dự đoán là churn.
            - `tuned_metrics`: là kết quả cuối cùng trên tập test sau khi đã chọn mô hình, tuning siêu tham số và tuning threshold.
            - `optimization_comparison`: cho thấy việc tuning threshold ảnh hưởng thế nào đến precision, recall và F1.
            - `classification_report`: cho biết metric riêng cho từng lớp 0 và 1.
            - `error_analysis`: giúp xem mô hình hay sai ở nhóm hợp đồng nào.

            Nếu thầy hỏi vì sao không chỉ tối ưu accuracy, có thể trả lời: vì khách churn thường là lớp ít hơn, nên mô hình đoán không churn nhiều vẫn có accuracy cao nhưng không hữu ích cho bài toán giữ chân khách hàng.


## 4. Minh họa kết quả

            Các biểu đồ dưới đây được lưu tự động trong `reports/figures/churn`.
            - Phân bố churn.
            - Tỷ lệ churn theo loại hợp đồng.
            - Monthly charges theo loại hợp đồng và churn.
            - Phân phối monthly charges theo churn.
            - Phân phối tenure.
            - Tỷ lệ churn theo phương thức thanh toán.
            - Tỷ lệ churn theo các dịch vụ khách hàng không sử dụng.
            - Biểu đồ so sánh F1-score giữa các mô hình.
            - Confusion matrix.
            - ROC curve.
            - Precision-Recall curve.
            - F1-score theo decision threshold.
            - Feature importance.


### Cách đọc phần biểu đồ Churn

            Khi trình bày, nên chia biểu đồ thành ba nhóm:
            - Nhóm EDA: tỷ lệ churn theo hợp đồng, phương thức thanh toán, tenure và monthly charges để hiểu nhóm khách nào rủi ro cao.
            - Nhóm đánh giá mô hình: confusion matrix, ROC curve và Precision-Recall curve để xem mô hình phân loại tốt hay chưa.
            - Nhóm tối ưu ngưỡng: biểu đồ F1 theo threshold cho thấy vì sao cần chọn ngưỡng khác 0.5.

            Điểm nên nhấn mạnh là bài churn không chỉ cần dự đoán đúng nhiều, mà còn cần bắt được khách có nguy cơ rời bỏ để doanh nghiệp kịp can thiệp.


### Nhận xét chuyên sâu từng biểu đồ Churn

| Biểu đồ | Nhận xét chi tiết | Thuật toán học được gì và có ích gì |
|---|---|---|
| `class_distribution` | Biểu đồ cho thấy tỷ lệ khách churn và không churn không cân bằng. Lớp không churn nhiều hơn, nên accuracy dễ gây hiểu nhầm nếu mô hình chỉ đoán theo lớp đa số. | Vì dữ liệu lệch lớp, dự án dùng `class_weight='balanced'`, F1, Recall và ROC-AUC. Điều này giúp thuật toán chú ý hơn đến lớp churn, không chỉ tối ưu dự đoán khách không churn. |
| `churn_rate_by_contract` | Khách hợp đồng month-to-month có tỷ lệ churn cao hơn nhiều so với hợp đồng một năm hoặc hai năm. Đây là tín hiệu nghiệp vụ rất mạnh. | Logistic Regression học hệ số cho từng loại hợp đồng sau One-Hot Encoding. Hợp đồng ngắn hạn làm tăng xác suất churn, giúp mô hình phát hiện nhóm khách cần giữ chân sớm. |
| `contract_vs_monthlycharges` | Kết hợp contract và monthly charges cho thấy rủi ro churn không chỉ đến từ một biến đơn lẻ. Khách hợp đồng linh hoạt và phí cao thường đáng chú ý hơn. | Mô hình học đồng thời nhiều biến nên có thể kết hợp tín hiệu chi phí và loại hợp đồng. Lợi ích là dự đoán sát thực tế hơn so với chỉ nhìn một biến riêng lẻ. |
| `monthlycharges_distribution` | Phân phối monthly charges của nhóm churn thường lệch về mức phí cao hơn. Điều này gợi ý giá dịch vụ có thể ảnh hưởng đến quyết định rời bỏ. | MonthlyCharges là biến số được chuẩn hóa trước khi đưa vào Logistic Regression. Nếu phí cao làm tăng xác suất churn, mô hình sẽ học hệ số tương ứng để nâng xác suất dự đoán. |
| `tenure_density` | Khách có tenure thấp thường churn nhiều hơn; khách gắn bó lâu có xu hướng ổn định hơn. Đây là một trong các tín hiệu mạnh nhất. | Logistic Regression học tenure như biến số liên tục. Khi tenure tăng, xác suất churn có thể giảm, giúp mô hình nhận ra khách mới là nhóm cần chăm sóc đặc biệt. |
| `churn_rate_by_payment_method` | Một số phương thức thanh toán có tỷ lệ churn cao hơn. Đây có thể phản ánh mức độ cam kết hoặc hành vi sử dụng dịch vụ khác nhau. | PaymentMethod được One-Hot Encoding để mô hình học rủi ro riêng từng nhóm. Điều này hữu ích cho chiến dịch chăm sóc khách hàng theo phương thức thanh toán. |
| `churn_rate_by_missing_services` | Khách không dùng các dịch vụ bổ sung như bảo mật, backup, hỗ trợ kỹ thuật có thể churn cao hơn. Điều này cho thấy mức độ gắn kết dịch vụ thấp. | Các biến dịch vụ bổ sung giúp mô hình hiểu “độ bám” của khách với hệ sinh thái sản phẩm. Doanh nghiệp có thể dùng insight này để đề xuất gói dịch vụ giữ chân. |
| `model_comparison_f1` | F1-score so sánh mô hình theo khả năng cân bằng giữa precision và recall. Mô hình có F1 cao hơn phù hợp hơn cho bài toán churn lệch lớp. | Logistic Regression đạt kết quả tốt và dễ giải thích. Đây là lựa chọn hợp lý cho bài coursework vì vừa có hiệu năng ổn, vừa trình bày được vì sao khách có nguy cơ churn. |
| `confusion_matrix` | Confusion matrix cho biết đúng/sai ở từng lớp. Cần đặc biệt chú ý false negative: khách thật sự churn nhưng mô hình dự đoán không churn. | Ma trận này giúp đánh giá tác động thực tế của mô hình. Nếu false negative cao, doanh nghiệp bỏ lỡ khách cần giữ chân; nếu false positive cao, chi phí chăm sóc tăng. |
| `roc_curve` | ROC curve càng xa đường chéo càng tốt. ROC-AUC cao cho thấy mô hình xếp hạng khách churn cao hơn khách không churn trên nhiều ngưỡng. | ROC-AUC đánh giá chất lượng xác suất, không phụ thuộc một threshold cố định. Điều này chứng minh mô hình có khả năng phân biệt hai lớp khá tốt. |
| `precision_recall_curve` | Precision-Recall curve quan trọng khi dữ liệu lệch lớp. Nó cho thấy muốn bắt được nhiều khách churn hơn thì phải chấp nhận bao nhiêu cảnh báo sai. | Đường cong này giúp chọn chiến lược kinh doanh: nếu ưu tiên không bỏ sót khách churn thì chọn recall cao; nếu muốn giảm chi phí chăm sóc sai thì cần precision cao hơn. |
| `threshold_vs_f1` | F1 thay đổi theo decision threshold, chứng minh ngưỡng mặc định 0.5 không nhất thiết tối ưu. Dự án chọn threshold tốt hơn để tăng F1. | Threshold tuning ảnh hưởng trực tiếp đến nhãn cuối cùng. Thuật toán tạo xác suất, còn threshold quyết định chuyển xác suất thành Yes/No. Đây là bước rất quan trọng trong bài toán churn. |
| `feature_importance` | Các biến như tenure, contract, total charges, internet service thường quan trọng. Đây đều là yếu tố có ý nghĩa nghiệp vụ rõ ràng. | Feature importance giúp giải thích mô hình: không chỉ dự đoán khách churn, mà còn chỉ ra lý do rủi ro. Điều này hữu ích cho vấn đáp và cho đề xuất giữ chân khách hàng. |


In [ ]:
for title, path in churn_results.figures.items():
                print(title, "->", path)
                display(Image(filename=path))


## 5. Kết luận

            Bài toán churn không nên chỉ đánh giá bằng `accuracy`, vì lớp khách hàng rời bỏ thường ít hơn lớp
            khách hàng không rời bỏ. Trong ngữ cảnh kinh doanh, bỏ sót khách hàng có nguy cơ churn có thể làm
            mất doanh thu, nên `Recall`, `F1-score` và `ROC-AUC` là các metric quan trọng hơn.

            Mô hình cuối cùng được tối ưu thêm bằng `decision threshold tuning`. Bảng so sánh trước/sau tuning
            cho thấy ngưỡng dự đoán có ảnh hưởng trực tiếp đến cân bằng giữa `Precision` và `Recall`.

            Về mặt nghiệp vụ, các biến như thời gian sử dụng dịch vụ, loại hợp đồng, dịch vụ internet,
            tổng chi phí và phương thức thanh toán là các yếu tố giải thích khả năng churn. Hướng phát triển tiếp
            theo là thử nghiệm thêm cost-sensitive learning, phân tích false negative, và điều chỉnh threshold theo
            chi phí kinh doanh thực tế.
